# Simple Colab Notebook: Train a Tiny London LLM with LoRA

This notebook is intentionally simple for educational use.
It fine-tunes a **very small Hugging Face model** on a small slice of the London 1800–1875 dataset and shows **custom inference**.


In [ ]:
# If running on Google Colab, install dependencies
!pip -q install -U transformers datasets peft accelerate bitsandbytes


In [ ]:
import os
import math
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel

# ---- Simple config ----
MODEL_NAME = "sshleifer/tiny-gpt2"     # tiny model for cheap Colab experiments
DATASET_NAME = "postgrammar/london-llm-1800"
MAX_SAMPLES = 2000                         # keep small for speed
MAX_LENGTH = 256
OUTPUT_DIR = "tiny_london_lora"
SEED = 42

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
# Load dataset (streaming so we don't download everything)
stream = load_dataset(DATASET_NAME, split="train", streaming=True)
rows = []
for i, row in enumerate(stream):
    rows.append(row)
    if i + 1 >= MAX_SAMPLES:
        break

dataset = Dataset.from_list(rows)
print(dataset)
print("Columns:", dataset.column_names)


In [ ]:
# Pick text column automatically (simple fallback logic)
text_col = "text" if "text" in dataset.column_names else dataset.column_names[0]
print("Using text column:", text_col)

# Train/validation split
dset = dataset.train_test_split(test_size=0.05, seed=SEED)
train_ds = dset["train"]
val_ds = dset["test"]
print(train_ds, val_ds)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_fn(batch):
    return tokenizer(
        batch[text_col],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(train_tok)


In [ ]:
# Load tiny base model + LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_model.resize_token_embeddings(len(tokenizer))

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"],  # works for GPT-2 style modules
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    eval_steps=100,
    save_steps=100,
    evaluation_strategy="steps",
    save_strategy="steps",
    warmup_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to", OUTPUT_DIR)


In [ ]:
# Quick evaluation: perplexity
metrics = trainer.evaluate()
print(metrics)
if "eval_loss" in metrics:
    print("Perplexity:", math.exp(metrics["eval_loss"]))


## Custom Inference Code

This loads the base model + your trained LoRA adapter and generates text.


In [ ]:
def load_inference_model(base_model_name=MODEL_NAME, adapter_dir=OUTPUT_DIR):
    tok = AutoTokenizer.from_pretrained(adapter_dir)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    base = AutoModelForCausalLM.from_pretrained(base_model_name)
    model = PeftModel.from_pretrained(base, adapter_dir)
    model = model.to(device)
    model.eval()
    return tok, model


def generate_text(prompt, tok, model, max_new_tokens=120, temperature=0.9, top_p=0.95):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tok.eos_token_id,
            eos_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)


inf_tok, inf_model = load_inference_model()
prompt = "In the year of our Lord 1834, the streets of London"
print(generate_text(prompt, inf_tok, inf_model))


### Notes for Colab
- If you get CUDA OOM, reduce `MAX_LENGTH` to 128 and batch size to 2.
- For a stronger (still small) model, try `distilgpt2` instead of `sshleifer/tiny-gpt2`.
- This notebook is for simplicity and learning, not best quality.
